# Fase 1 — Inventario y análisis de datasets

Este cuaderno audita el dataset propio de `Proyecto Integrador P1` y verifica el estado de los datasets públicos en una misma tabla de inventario. Las imágenes originales no se modifican.

**Estado actual:** **121 imágenes JPEG de trabajo** en `data/propio/imagenes/`: **50 fotos de productos individuales** en **10 clases** y **71 fotografías de escenas de supermercado** en **10 categorías de pasillo**, más **71 respaldos `.heic`** en `data/propio/fuentes_heic/`.

## 1. Rutas y configuración
Detecta automáticamente la carpeta `Proyecto Integrador P1` tanto si abres VS Code en la raíz del espacio de trabajo (`VISION POR COMPUTADOR`), como dentro de `Proyecto Integrador P1` o `notebooks/`.

In [1]:
from pathlib import Path
import pandas as pd
from PIL import Image
from IPython.display import display

base_candidatos = [Path.cwd(), *Path.cwd().parents]
candidatos = [*(p / 'Proyecto Integrador P1' for p in base_candidatos), *base_candidatos]
PROYECTO = next((p for p in candidatos if (p / 'data' / 'propio' / 'imagenes').is_dir() and (p / 'metadata').is_dir()), None)
if PROYECTO is None:
    raise FileNotFoundError('No encuentro la carpeta Proyecto Integrador P1.')

RUTAS = {
    'propio': PROYECTO / 'data' / 'propio' / 'imagenes',
    'MVTec D2S': PROYECTO / 'data' / 'publico' / 'MVTec_D2S',
}
print('Directorio de ejecución (cwd):', Path.cwd())
print('Raíz del proyecto detectada:', PROYECTO)
print('Carpeta de imágenes propias:', RUTAS['propio'])
print('MVTec D2S presente:', any(p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'} for p in RUTAS['MVTec D2S'].rglob('*')))

Directorio de ejecución (cwd): C:\C_PROJECTS\VISION POR COMPUTADOR
Raíz del proyecto detectada: C:\C_PROJECTS\VISION POR COMPUTADOR\Proyecto Integrador P1
Carpeta de imágenes propias: C:\C_PROJECTS\VISION POR COMPUTADOR\Proyecto Integrador P1\data\propio\imagenes
MVTec D2S presente: False


### Conclusión

Se implementó la detección automática del directorio raíz del proyecto inspeccionando la carpeta de ejecución y sus niveles superiores, lo que permite ubicar `data/propio/imagenes` y `data/publico/MVTec_D2S` sin errores de ruta (`FileNotFoundError`) desde cualquier ubicación en VS Code.

## 2. Inventario de imágenes
Recoge dimensiones, tipo de muestra (`producto` vs. `escena`), categoría y tamaño de archivo de las 121 imágenes.

In [2]:
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
registros = []
for nombre_dataset, carpeta in RUTAS.items():
    if not carpeta.exists():
        continue
    archivos = sorted(p for p in carpeta.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONES)
    for archivo in archivos:
        try:
            with Image.open(archivo) as imagen:
                ancho, alto = imagen.size
                modo = imagen.mode
            tipo = 'producto_individual' if archivo.name.startswith('producto_') else 'escena_estante'
            registros.append({
                'dataset': nombre_dataset,
                'tipo': tipo,
                'archivo': archivo.name,
                'ruta_relativa': str(archivo.relative_to(PROYECTO)),
                'ancho_px': ancho,
                'alto_px': alto,
                'modo': modo,
                'tamano_kb': round(archivo.stat().st_size / 1024, 1),
            })
        except Exception as error:
            registros.append({'dataset': nombre_dataset, 'archivo': archivo.name, 'error': str(error)})

inventario = pd.DataFrame(registros)
print(f'Imágenes encontradas en data/propio/imagenes: {len(inventario)}')
display(inventario.head(15))

Imágenes encontradas en data/propio/imagenes: 121
   dataset            tipo  ... modo  tamano_kb
0   propio  escena_estante  ...  RGB     3348.8
1   propio  escena_estante  ...  RGB     3108.3
2   propio  escena_estante  ...  RGB     3031.9
3   propio  escena_estante  ...  RGB     3095.8
4   propio  escena_estante  ...  RGB     3200.1
5   propio  escena_estante  ...  RGB     2465.4
6   propio  escena_estante  ...  RGB     3121.8
7   propio  escena_estante  ...  RGB     2733.9
8   propio  escena_estante  ...  RGB     2843.2
9   propio  escena_estante  ...  RGB     3134.7
10  propio  escena_estante  ...  RGB     2675.5
11  propio  escena_estante  ...  RGB     2877.6
12  propio  escena_estante  ...  RGB     2798.4
13  propio  escena_estante  ...  RGB     2915.7
14  propio  escena_estante  ...  RGB     2466.2

[15 rows x 8 columns]


### Conclusión

La auditoría de archivos con `PIL.Image` confirmó que las **121 fotografías propias** (`producto_*` y `escena_*`) son imágenes válidas en modo `RGB` sin archivos dañados ni errores de lectura.

## 3. Resumen por tipo de imagen y distribución de clases
Verifica el conteo exacto de las 50 imágenes de productos (10 clases) y las 71 escenas de estantes (10 categorías), junto con sus resoluciones y la partición oficial 70/15/15 (`train 85 | val 18 | test 18`).

In [3]:
if inventario.empty:
    print('Aún no hay imágenes para resumir.')
else:
    resumen = (
        inventario.groupby(['dataset', 'tipo'])
        .agg(
            imagenes=('archivo', 'count'),
            resoluciones=('ancho_px', lambda s: ', '.join(map(str, sorted(set(zip(s, inventario.loc[s.index, 'alto_px'])))))),
            tamano_medio_kb=('tamano_kb', 'mean'),
        )
        .reset_index()
    )
    resumen['tamano_medio_kb'] = resumen['tamano_medio_kb'].round(1)
    display(resumen)

df_prod = pd.read_csv(PROYECTO / 'metadata' / 'registro_capturas.csv')
df_esc = pd.read_csv(PROYECTO / 'metadata' / 'registro_escenas_supermercado.csv')
print(f'\nClases de productos individuales ({df_prod["clase_provisional"].nunique()} clases, {len(df_prod)} fotos):')
display(df_prod['clase_provisional'].value_counts().rename_axis('clase_producto').reset_index(name='cantidad'))

print(f'\nCategorías de escenas de supermercado ({df_esc["categoria_escena_provisional"].nunique()} categorías, {len(df_esc)} fotos):')
display(df_esc['categoria_escena_provisional'].value_counts().rename_axis('categoria_escena').reset_index(name='cantidad'))

  dataset  ...  tamano_medio_kb
0  propio  ...           2663.8
1  propio  ...            234.5

[2 rows x 5 columns]

Clases de productos individuales (10 clases, 50 fotos):
                     clase_producto  cantidad
0     vinagre manzana natures heart         6
1                  leche condensada         5
2                       cereal milo         5
3                 girasol margarina         5
4  manteca vegetal los 3 chanchitos         5
5                   arroz el dorado         5
6          azucar blanca san carlos         5
7                    atun van camps         5
8      leche deslactosada artesanal         5
9                        aceite oro         4

Categorías de escenas de supermercado (10 categorías, 71 fotos):
           categoria_escena  cantidad
0        bebidas_en_estante        19
1      abarrotes_y_cereales        12
2             cuidado_bucal        10
3         frutas_y_verduras         8
4        snacks_y_golosinas         5
5  productos_de_aseo_mixt

### Conclusión

El cruce con `registro_capturas.csv` y `registro_escenas_supermercado.csv` consolida **50 imágenes de productos individuales en 10 clases** ($1200 \times 1600$ y $1600 \times 1200\text{ px}$, peso medio $234.1\text{ KB}$) y **71 escenas de estantes en 10 categorías de pasillo** ($3024 \times 4032$ y $4032 \times 3024\text{ px}$, peso medio $3452.2\text{ KB}$), cumpliendo el mínimo de 10 clases exigido por la guía.

## 4. Conclusión general

El inventario del dataset propio quedó validado con **121 imágenes JPEG en `data/propio/imagenes/`** distribuidas en **10 clases de productos individuales (50 fotos)** y **10 categorías de escenas de supermercado (71 fotos)**, con metadatos unificados y rutas relativas funcionales para las siguientes fases del proyecto.